# Auditoria das estratégias de pré-processamento P0, P1 e P2

A auditoria verifica a implementação das transformações em oito imagens fixas da validação, por inspeção visual e medidas de luminância. A comparação de desempenho é feita nas execuções de treinamento.

## Configurações de pré-processamento

- **P0 — RGB:** referência sem correção de cor.
- **P1 — Shades-of-Gray (`p=6`):** correção de cor baseada na estimativa do iluminante.
- **P2 — CLAHE em L\*:** realce local de contraste no canal de luminosidade do espaço CIELAB.

P1 e P2 retornam RGB. Eles não são combinados nesta triagem.

## Representação de entrada

As três configurações mantêm a entrada em RGB. Em P2, a conversão para CIELAB permite modificar o canal L\* antes do retorno ao espaço RGB.

## 1. Montar o Drive e localizar o projeto

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys

try:
    from google.colab import drive

    drive.mount("/content/drive")
except ImportError:
    print("Execução fora do Colab.")

candidates = [
    Path("/content/drive/MyDrive/TCC/Projeto_TCC_Lucas"),
    Path("/content/drive/MyDrive/Projeto_TCC_Lucas"),
]
PROJECT_ROOT = next((p for p in candidates if p.is_dir()), Path.cwd())
if not (PROJECT_ROOT / "configs/preprocessing_audit.yaml").is_file():
    raise FileNotFoundError("Configuração da auditoria de pré-processamento não encontrada.")
os.chdir(PROJECT_ROOT)
SOURCE_ROOT = PROJECT_ROOT / "src"
os.environ["PYTHONPATH"] = str(SOURCE_ROOT) + os.pathsep + os.environ.get("PYTHONPATH", "")
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))
print("Projeto:", PROJECT_ROOT)
print("Código-fonte:", SOURCE_ROOT)

## 2. Instalar as dependências registradas

In [ ]:
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"],
    text=True,
    capture_output=True,
)
print(result.stdout)
print(result.stderr)
result.check_returncode()

## 3. Copiar os dados para o cache temporário

In [ ]:
DATA_ROOT = Path("/content/tcc_data") if Path("/content").is_dir() else PROJECT_ROOT
if DATA_ROOT != PROJECT_ROOT:
    shutil.copytree(PROJECT_ROOT / "Dataset", DATA_ROOT / "Dataset", dirs_exist_ok=True)
print("Dados:", DATA_ROOT / "Dataset")

## 4. Conferir a configuração pré-registrada

In [ ]:
import yaml
from IPython.display import display

CONFIG = yaml.safe_load(
    (PROJECT_ROOT / "configs/preprocessing_audit.yaml").read_text(encoding="utf-8")
)
display(CONFIG)
assert list(CONFIG["candidates"]) == ["P0", "P1", "P2"]
assert CONFIG["gate"]["winner_requires_training_validation_metrics"]

## 5. Testar as funções isoladamente

Os testes verificam determinismo, formato RGB `uint8`, preservação da máscara e rejeição de uma combinação P1+P2 não autorizada.

In [ ]:
test_environment = os.environ.copy()
test_environment["PYTHONPATH"] = (
    str(PROJECT_ROOT / "src") + os.pathsep + test_environment.get("PYTHONPATH", "")
)
result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "tests/test_preprocessing.py"],
    cwd=PROJECT_ROOT,
    env=test_environment,
    text=True,
    capture_output=True,
)
print(result.stdout)
print(result.stderr)
result.check_returncode()

## 6. Gerar comparação visual e diagnósticos

As duas últimas colunas do painel mostram a intensidade da mudança em relação a P0. Elas mostram **onde a imagem mudou**, não a qualidade da segmentação.

In [ ]:
command = [
    sys.executable,
    "scripts/audit_preprocessing.py",
    "--project-root",
    str(PROJECT_ROOT),
    "--data-root",
    str(DATA_ROOT),
    "--config",
    "configs/preprocessing_audit.yaml",
]
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in process.stdout:
    print(line, end="")
returncode = process.wait()
print("returncode=", returncode)
if returncode != 0:
    raise subprocess.CalledProcessError(returncode, command)

## 7. Localizar e visualizar o relatório produzido

In [ ]:
from PIL import Image
import pandas as pd

AUDIT_ROOT = PROJECT_ROOT / CONFIG["runtime"]["output_root"]
OUTPUT_DIR = max((p for p in AUDIT_ROOT.iterdir() if p.is_dir()), key=lambda p: p.stat().st_mtime)
print("Saídas:", OUTPUT_DIR)
display(Image.open(OUTPUT_DIR / "preprocessing_panel.png"))
display(Image.open(OUTPUT_DIR / "luminance_histograms.png"))
diagnostics = pd.read_csv(OUTPUT_DIR / "preprocessing_diagnostics.csv")
display(diagnostics.groupby("candidate").mean(numeric_only=True))

## Como interpretar

- P1 deve alterar principalmente o equilíbrio entre os canais RGB; correções extremas ou saturação são sinais de alerta.
- P2 deve redistribuir a luminância e revelar contraste local; halos, textura exagerada ou reflexos amplificados são sinais de alerta.
- As diferenças visuais caracterizam o efeito das transformações. O desempenho de segmentação é avaliado após o treinamento.

## Artefatos da execução

Os arquivos registrados incluem `preprocessing_audit_summary.json`, `preprocessing_panel.png`, `luminance_histograms.png` e `preprocessing_diagnostics.csv`. A auditoria precede os treinamentos P1 e P2. Teste e FUSeg continuam fechados.